In [1]:
%load_ext autoreload
%autoreload 2
%cd /home/albin/egna_proj/block_puzzle_rl/

/home/albin/egna_proj/block_puzzle_rl


/home/albin/egna_proj/block_puzzle_rl/.venv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import numpy as np
import torch
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from gymnasium import spaces
from stable_baselines3.common.callbacks import BaseCallback
import numpy as np
from collections import deque
from stable_baselines3.common.callbacks import CallbackList
from sb3_contrib import MaskablePPO
from game.plot_game import render_text
from sb3_contrib.common.maskable.utils import get_action_masks
from sb3_contrib.common.wrappers import ActionMasker
import os
from datetime import datetime
from game.block_puzzle_env import BlockPuzzleEnv
from agent.utils import encode_state, encode_state_cnn_modern, encode_state_cnn_channeled
from sb3_utils.callbacks import AveragedMetricsCallback, SaveEveryNTimestepsCallback, UnfreezeCallback
import torch
import torch.nn as nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from game.plot_game import render_text_with_blocks
from game.block import random_block

In [3]:
class DiscreteActionWrapper(gym.Env):
    def __init__(self, raw_env):
        super().__init__()
        self.raw_env = raw_env
        self.original_action_space = raw_env.action_space  # Should be MultiDiscrete
        self.obs_space = self.raw_env.observation_space
        
        dummy_obs, _ = self.raw_env.reset()
        grid = encode_state_cnn_channeled(dummy_obs)

        # Flatten MultiDiscrete([a, b, c]) → Discrete(a * b * c)
        self.observation_space = spaces.Box(low=0.0, high=1.0, shape=grid.shape, dtype=np.float32)
        self.action_space = spaces.Discrete(np.prod(self.original_action_space.nvec))

    def reset(self, **kwargs):
        obs_dict, _ = self.raw_env.reset(**kwargs)
        grid = encode_state_cnn_channeled(obs_dict)
        return grid.astype(np.float32), {}

    def step(self, flat_action):
        a0, a1, a2 = np.unravel_index(flat_action, self.original_action_space.nvec)
        action = (int(a0), int(a1), int(a2))
        (obs_dict, _), reward, terminated, truncated, info = self.raw_env.step(action)
        grid = encode_state_cnn_channeled(obs_dict)
        return grid.astype(np.float32), reward, terminated, truncated, info

    def render(self, **kwargs):
        return self.raw_env.render(**kwargs)



In [4]:
raw_env = BlockPuzzleEnv(width=8, height=10, num_blocks=3, block_function=random_block)
wrapped_env = DiscreteActionWrapper(raw_env)
def mask_fn(env):
    return env.raw_env.game.compute_action_mask()
masked_env = ActionMasker(wrapped_env, mask_fn)
check_env(masked_env, warn=True) 

/home/albin/egna_proj/block_puzzle_rl/.venv/lib/python3.10/site-packages/stable_baselines3/common/env_checker.py:55: UserWarning: It seems that your observation  is an image but its `dtype` is (float32) whereas it has to be `np.uint8`. If your observation is not an image, we recommend you to flatten the observation to have only a 1D vector
  warnings.warn(
/home/albin/egna_proj/block_puzzle_rl/.venv/lib/python3.10/site-packages/stable_baselines3/common/env_checker.py:63: UserWarning: It seems that your observation space  is an image but the upper and lower bounds are not in [0, 255]. Because the CNN policy normalize automatically the observation you may encounter issue if the values are not in that range.
  warnings.warn(
/home/albin/egna_proj/block_puzzle_rl/.venv/lib/python3.10/site-packages/stable_baselines3/common/env_checker.py:76: UserWarning: The minimal resolution for an image is 36x36 for the default `CnnPolicy`. You might need to use a custom features extractor cf. https://st

In [5]:
import torch.nn as nn
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class BlockPuzzleFeaturesExtended(BaseFeaturesExtractor):
    """
    Custom feature extractor for single-array (multi-channel grid) input.
    Applies a CNN to the input grid and then an MLP to the flattened output.
    """
    def __init__(self, observation_space, cnn_channels=[16,32,32,64,64]):
        super().__init__(observation_space, features_dim=1)  # features_dim will be set below
        grid_shape = observation_space.shape  # (C, H, W)
        C, H, W = grid_shape
        cnn_layers = []
        in_channels = C
        for out_channels in cnn_channels:
            cnn_layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1))
            cnn_layers.append(nn.ReLU())
            in_channels = out_channels
        cnn_layers.append(nn.Flatten())
        self.cnn = nn.Sequential(*cnn_layers)
        with torch.no_grad():
            dummy_grid_obs = torch.zeros(1, C, H, W)
            cnn_out_features = self.cnn(dummy_grid_obs)
            cnn_out_size = cnn_out_features.shape[1]
        self.joint_net = nn.Sequential(
            nn.Linear(cnn_out_size, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
        )
        self._features_dim = 128
    def forward(self, observations) -> torch.Tensor:
        # observations: (batch, C, H, W)
        x = self.cnn(observations)
        return self.joint_net(x)

In [6]:
def lr_warmup_schedule(progress_remaining) -> float:
    """
    SB3 passes in `progress_remaining` which goes from 1.0 → 0.0 over the entire learn() call.
    We want:
      - first 10% (progress 1.0 → 0.9): lr goes 1e-5 → 1e-3
      - remaining 90% (progress 0.9 → 0.0): lr goes 1e-3 → 1e-4
    """
    max_lr = 1e-3
    init_lr = 1e-5
    final_lr = 1e-4
    warmup_percent = 0.05  # 5% of the training time is warm-up

    # Calculate how far we are into training [0.0 .. 1.0]
    t = 1.0 - progress_remaining

    if t < warmup_percent:
        # warm-up phase: 0 → 0.05
        return init_lr + (max_lr - init_lr) * (t / warmup_percent)
    else:
        # decay phase: 0.05 → 1.0
        decay_t = (t - warmup_percent) / (1.0 - warmup_percent)  # remap [0.05..1.0] → [0..1]
        return max_lr + (final_lr - max_lr) * decay_t

In [7]:
def make_env():
    def _init():
        raw_env = BlockPuzzleEnv(width=8, height=10, num_blocks=3, block_function=random_block)
        wrapped_env = DiscreteActionWrapper(raw_env)

        def mask_fn(env):
            return env.raw_env.game.compute_action_mask()

        masked_env = ActionMasker(wrapped_env, mask_fn)
        monitored_env = Monitor(masked_env)
        return monitored_env

    return _init

n_envs = 32
vec_env = SubprocVecEnv([make_env() for _ in range(n_envs)])

policy_kwargs = dict(
    features_extractor_class  = BlockPuzzleFeaturesExtended,
    features_extractor_kwargs = dict(
        cnn_channels = [16, 32, 32, 64, 64],
    ),
)

model = MaskablePPO(
    policy            = "CnnPolicy",
    env               = vec_env,
    learning_rate     = 8e-4,
    n_steps           = 256,
    batch_size        = 4096,
    gamma             = 0.90,
    device            = "cuda",
    verbose           = 1,
    tensorboard_log   = "./sb3_logs/",
    policy_kwargs     = policy_kwargs,
)

callback_list = CallbackList([
    AveragedMetricsCallback(), 
    SaveEveryNTimestepsCallback(save_freq=250_000, save_path="./crash_backup_save/", name="backup_save", with_time=False, with_timesteps=False),
])

load_weights = True
weight_path = "crash_backup_save/backup_save__"
if load_weights:
    model.set_parameters(weight_path, exact_match=False)


model.learn(total_timesteps=100_000_000, callback=callback_list, tb_log_name="PPO_CNN_MASKED_CHANNELS")
model.save("sb3_block_ppo_cnn_modern_masked_channels")

Using cuda device
Logging to ./sb3_logs/PPO_CNN_MASKED_CHANNELS_6
-----------------------------------
| custom/              |          |
|    cleared_lines_avg | 10.4     |
|    invalid_moves_avg | 0        |
|    move_count_avg    | 38.8     |
| rollout/             |          |
|    ep_len_mean       | 38.8     |
|    ep_rew_mean       | 0.628    |
| time/                |          |
|    fps               | 3302     |
|    iterations        | 1        |
|    time_elapsed      | 2        |
|    total_timesteps   | 8192     |
-----------------------------------
------------------------------------------
| custom/                 |              |
|    cleared_lines_avg    | 9.42         |
|    invalid_moves_avg    | 0            |
|    move_count_avg       | 36.2         |
| rollout/                |              |
|    ep_len_mean          | 36.2         |
|    ep_rew_mean          | -0.318       |
| time/                   |              |
|    fps                  | 3118         |


KeyboardInterrupt: 

In [ ]:
model.learn(total_timesteps=100_000_000, callback=callback_list, tb_log_name="PPO_CNN_MASKED_CHANNELS")

In [ ]:
# callback_list = CallbackList([
#     AveragedMetricsCallback(), 
#     SaveEveryNTimestepsCallback(save_freq=1_000_000, save_path="./sb3_model_saves/", name="sb3_block_ppo_mlp_masked_rewards2"),
#     SaveEveryNTimestepsCallback(save_freq=100_000, save_path="./crash_backup_save/", name="backup_save", with_time=False, with_timesteps=False),
# ])

# for i in range(0, 50):
#     if i != 0:
#         backup_path = "crash_backup_save/backup_save__"
#         model.set_parameters(backup_path, exact_match=True)

#     try:
#         model.learn(total_timesteps=30_000_001, callback=callback_list, tb_log_name="PPO_MLP_MASKED")
#     except Exception as e:
#         pass

In [22]:
single_env = DiscreteActionWrapper(BlockPuzzleEnv(width=8, height=10, num_blocks=3, block_function=random_block))
obs, _ = single_env.reset()
done = False
total_reward = 0
step = 0

while not done:
    # Get valid action mask
    mask = single_env.raw_env.game.compute_action_mask()
        
    # Use model.predict with action_masks for MaskablePPO
    action, _ = model.predict(obs, action_masks=mask, deterministic=True)
    
    obs, reward, terminated, truncated, info = single_env.step(action)
    total_reward += reward
    done = terminated or truncated
    step += 1
    
    game_grid = single_env.raw_env.game.grid.get_game_grid()
    game_blocks = [block.grid() for block in single_env.raw_env.game.block_queue]

    render_text_with_blocks(game_grid, game_blocks)
    print()

print(f"Evaluation finished in {step} steps, total reward: {total_reward}")


□ □ □ □ □ □ □ □ | ■ ■ □ □  □ □ □ □  ■ □ □ □
□ □ □ □ □ □ □ □ | □ ■ ■ □  □ □ □ □  ■ □ □ □
□ □ □ □ □ □ □ □ | □ □ □ □  □ □ □ □  □ □ □ □
■ □ □ □ □ □ □ □ | □ □ □ □  □ □ □ □  □ □ □ □
■ □ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |

□ □ □ □ □ □ □ □ | ■ ■ □ □  □ □ □ □  □ □ □ □
□ □ □ □ □ □ □ □ | □ ■ ■ □  □ □ □ □  □ □ □ □
□ □ □ □ □ □ □ □ | □ □ □ □  □ □ □ □  □ □ □ □
■ ■ □ □ □ □ □ □ | □ □ □ □  □ □ □ □  □ □ □ □
■ ■ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |

□ □ □ □ □ □ □ □ | ■ ■ □ □  □ ■ □ □  □ ■ □ □
□ □ □ □ □ □ □ □ | □ ■ □ □  ■ ■ □ □  ■ ■ □ □
□ □ □ □ □ □ □ □ | □ □ □ □  ■ □ □ □  □ □ □ □
■ ■ ■ ■ □ □ □ □ | □ □ □ □  □ □ □ □  □ □ □ □
■ ■ □ ■ ■ □ □ □ |
□ □ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |
□ □ □ □ □ □ □ □ |

□ □ □ □ □ □ □ □ | ■ ■ □ □  □ □ □ □  □ ■ □ □
□ □ □ □ □ □ □ □ | □ ■ □ □  □ □ □ □  ■ ■ □ □
□ □ □ □ □ □ □ □ | □ □ □ □  □ □ □ □  □ □ □ □
■ ■ ■ ■ □ □ □

In [ ]:
game_grid